In [30]:
## IMPORTS AND SETUP
# Imports
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import datetime
import logging
import warnings
import wandb
import shutil
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from wandb.integration.keras import WandbMetricsLogger
from dotenv import load_dotenv
from tensorflow.keras.applications import VGG16
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight


# Force GPU usage
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow is using GPU: ", tf.test.is_gpu_available())
print("Devices: ", tf.config.list_physical_devices())
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        pass

# Hide TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0=default, 1=info, 2=warning, 3=error
tf.get_logger().setLevel(logging.ERROR)
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
tf.debugging.set_log_device_placement(False)
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.FATAL)

# WanDB init
load_dotenv("/tf/projet/.env")
WANDB_API_KEY = os.getenv("API_KEY")
wandb.login(key=WANDB_API_KEY, relogin=True)
print(WANDB_API_KEY)

I0000 00:00:1744373506.238050      38 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744373506.238202      38 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744373506.238235      38 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744373506.239597      38 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-11 12:11:46.239625: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.

Num GPUs Available:  1
TensorFlow is using GPU:  True
Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
bd46fcb69d8110468ee6567789285061c4baf1db


In [31]:
## PARAMETERS
seed = 123
data_format_fix = False # Set to True to fix data formats
data_visualization = False # Set to True to visualize data
load_model = False # Set to True to load existing models

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset" # Path to the raw data folder
excluded_data_folders = ["Dataset Livrable 2"] # Folders to exclude from the dataset
batch_size = 32 # Batch size for dataset loading
img_height = 180 # Image height for dataset loading
img_width = 180 # Image width for dataset loading

# Dataset split parameters
train_split = 0.8 # Proportion of the dataset to use for training
val_split = 0.1 # Proportion of the dataset to use for validation
test_split = 0.1 # Proportion of the dataset to use for testing

# Models loading parameters
models_folder = "/tf/projet/Livrable 1/models" # Path to the models folder
excluded_models = ["CNN.keras"] # Models to exclude from testing

# Models creation parameters
transfer_learning = True # Set to True to create a VGG16 model with custom top, False to create a custom model
target_binary_class_name = "Painting" # Name of the target binary class for binary classification, None for multi-class
class_weight = False # Set to True to use class weights for imbalanced datasets

# Models training parameters
learning_rate = 0.001
epochs = 10
# history_save_path = models_folder + "/history.json" TODO: à revoir
save_path = None # Keep None, otherwise will overide existing model
trainable = False # Set to True to train the model

project_name = "Leyanda"
model_arch = "CNN"
entity = "tom-antoine-cesi" # WANDB entity name

image_size = f"{img_height}x{img_width}"



In [ ]:
def generate_model_name(
    project_name,
    model_arch,
    target_class=None,
    transfer_learning=False,
    epoch=None,
    batch_size=None,
    custom_tag=None
):
    """
    Generate a standardized name for ML models.

    Parameters:
    - project_name (str): Name of the project
    - model_arch (str): Architecture name (CNN, ResNet, etc.)
    - target_class (str, optional): Target class for binary classification
    - transfer_learning (bool, optional): Whether transfer learning was used
    - epoch (int, optional): Number of epochs
    - batch_size (int, optional): Batch size used for training
    - custom_tag (str, optional): Additional custom tag
    Returns:
    - str: Standardized model name
    """
    components = []

    # Add mandatory components
    components.append(project_name)
    components.append(model_arch)

    # Add optional components if they exist
    if target_class:
        components.append(f"bin_{target_class}")
    else:
        components.append("multi")

    if transfer_learning:
        components.append("transfer")

    if epoch is not None:
        components.append(f"e{epoch}")

    if batch_size is not None:
        components.append(f"b{batch_size}")

    if custom_tag:
        components.append(custom_tag)

    # Join all components with underscores
    model_name = "_".join(components)

    return model_name


# Exemple d'utilisation avec tes paramètres actuels
model_name = generate_model_name(
    project_name=project_name,
    model_arch=model_arch,
    target_class=target_binary_class_name if target_binary_class_name else None,
    transfer_learning=transfer_learning,
    epoch=epochs,
    batch_size=batch_size
)

print(f"Generated model name: {model_name}")

# Fonction pour générer le chemin complet du modèle
def get_model_path(model_name, models_folder):
    """
    Generate the full path for a model.

    Args:
        model_name (str): Name of the model
        models_folder (str): Path to the models folder

    Returns:
        str: Full path to the model file
    """
    return f"{models_folder}/{model_name}.keras"

# Exemple d'utilisation pour générer le chemin complet
model_path = get_model_path(model_name, models_folder)
print(f"Model full path: {model_path}")

In [33]:
## DATA PREPARATION FUNCTIONS
# Data format fixes
def data_formats_fixes(raw_data_path):
    """
    Walks through a directory to detect and remove problematic image files.
    Removes:
    - Files that are not actually JPG format
    - Corrupted or unreadable images
    Converts:
    - Invalid shape files to RGB format
    Parameters:
    - raw_data_path: Path to the raw data folder.
    """
    print(f"--Starting data format fixes--")

    stats = {
        "processed": 0,
        "wrong_format_removed": 0,
        "invalid_shape_converted": 0, # Includes grayscale
        "corrupted_removed": 0,
        "valid_images": 0
    }

    total_files = 0
    for root, dirs, files in os.walk(raw_data_path):
        for file in files:
            if os.path.splitext(file)[1].lower() == '.jpg':
                total_files += 1

    with tqdm(total=total_files, desc="Checking images") as pbar:
        for root, dirs, files in os.walk(raw_data_path):
            for file in files:
                file_path = os.path.join(root, file)
                _, extension = os.path.splitext(file)

                if extension.lower() != '.jpg':
                    continue

                stats["processed"] += 1
                pbar.update(1)

                try:
                    with open(file_path, 'rb') as f:
                        header = f.read(4)

                    if header[:2] != b'\xff\xd8':  # Not a valid JPEG header
                        os.remove(file_path)
                        stats["wrong_format_removed"] += 1
                        continue

                    try:
                        with Image.open(file_path) as img:
                            if img.mode != 'RGB':
                                img_rgb = img.convert('RGB')
                                img_rgb.save(file_path, 'JPEG', quality=95)
                                stats["invalid_shape_converted"] += 1
                                stats["valid_images"] += 1
                                continue

                            stats["valid_images"] += 1

                    except Exception as e:
                        os.remove(file_path)
                        stats["corrupted_removed"] += 1

                except Exception as e:
                    os.remove(file_path)
                    stats["corrupted_removed"] += 1

    print(f"\nSummary:")
    print(f"Files processed: {stats['processed']}")
    print(f"Wrong format files removed: {stats['wrong_format_removed']}")
    print(f"Invalid shape files converted: {stats['invalid_shape_converted']}")
    print(f"Corrupted files removed: {stats['corrupted_removed']}")
    print(f"Valid images remaining: {stats['valid_images']}")
    print(f"Data format fixes completed.")


# Dataset assembly
def dataset_assembly(raw_data_path, subfolders, batch_size, img_height, img_width, seed):
    """
    Assembles a dataset from a folder structure.
    Parameters:
    - raw_data_path: Path to the raw data folder.
    - subfolders: List of subfolders to include in the dataset.
    Returns:
    - dataset: A TensorFlow dataset object.
    """
    print(f"--Starting dataset assembly--")
    try:
        dataset = tf.keras.utils.image_dataset_from_directory(
            raw_data_path,
            labels="inferred",
            label_mode="int",
            class_names=subfolders,
            color_mode="rgb",
            batch_size=batch_size,
            image_size=(img_height, img_width),
            shuffle=True,
            seed=seed,
            validation_split=None,
            subset=None,
            interpolation="bilinear",
            follow_links=False
        )

        class_names = dataset.class_names
        print(f"Detected classes: {class_names}")

        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

        for images, labels in dataset.take(1):
            print(f"Images batch shape : {images.shape}")

        print(f"Dataset assembly completed.")
        return dataset, class_names

    except Exception as e:
        print(f"Error creating dataset: {e}")


# Dataset split
def dataset_split(dataset, train_split=0.8, val_split=0.1, batch_size=1000, seed=123):
    """
    Splits a dataset into training, validation, and test sets.
    Parameters:
    - dataset: The dataset to split.
    - train_split: Proportion of the dataset to use for training.
    - val_split: Proportion of the dataset to use for validation.
    - test_split: Proportion of the dataset to use for testing.
    Returns:
    - train_ds: Training dataset.
    - val_ds: Validation dataset.
    - test_ds: Test dataset.
    """
    print(f"--Starting dataset split--")
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()  # Faster than len(dataset)
    print(f"Dataset size: {dataset_size}")
    train_size = int(train_split * dataset_size)
    val_size = int(val_split * dataset_size)
    test_size = dataset_size - train_size - val_size

    print(f"Creating training set of size ~{train_size*batch_size}...")
    train_ds = dataset.take(train_size)
    remaining_ds = dataset.skip(train_size)
    print(f"Creating validation set of size ~{val_size*batch_size}...")
    val_ds = remaining_ds.take(val_size)
    print(f"Creating test set of size ~{test_size*batch_size}...")
    test_ds = remaining_ds.skip(val_size)

    print(f"Dataset split completed.")
    return train_ds, val_ds, test_ds


# Convert dataset to binary structure
def convert_to_binary_dataset_structure(source_path, target_binary_class, output_path=None):
    """
    Creates a folder with two subfolders:
    - target_binary_class: contains images belonging to the target class
    - not_target_binary_class: contains images from all other classes
    Parameters:
    - source_path: path to the original dataset folder (with n subfolders/classes)
    - target_binary_class: name of the positive class
    - output_path: path to the output folder (optional). If None, creates Dataset_binary_<target_binary_class>
    Returns:
    - binary_path: path to the folder containing the binary structure
    """
    print(f"--Converting dataset to binary format--")

    if not os.path.isdir(source_path):
        raise ValueError(f"Source path '{source_path}' does not exist or is not a directory.")

    class_dirs = [d for d in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, d))]

    if target_binary_class not in class_dirs:
        raise ValueError(f"Class '{target_binary_class}' not found in source directory. Found classes: {class_dirs}")

    if output_path is None:
        parent_dir = os.path.dirname(source_path)
        output_path = os.path.join(parent_dir, f"Dataset_binary_{target_binary_class}")
    if os.path.exists(output_path):
        shutil.rmtree(output_path)

    pos_dir = os.path.join(output_path, target_binary_class)
    neg_dir = os.path.join(output_path, f"not_{target_binary_class}")
    os.makedirs(pos_dir, exist_ok=True)
    os.makedirs(neg_dir, exist_ok=True)

    for class_name in class_dirs:
        src_dir = os.path.join(source_path, class_name)
        dst_dir = pos_dir if class_name == target_binary_class else neg_dir

        for filename in os.listdir(src_dir):
            src_file = os.path.join(src_dir, filename)
            if os.path.isfile(src_file):
                dst_file = os.path.join(dst_dir, filename)
                shutil.copy2(src_file, dst_file)

    print(f"Binary dataset created in: {output_path}")
    print(f"Classes: {target_binary_class}, not_{target_binary_class}")
    return output_path

In [34]:
## DATA PREPARATION WORKFLOW
if target_binary_class_name is not None:
    raw_data_path = convert_to_binary_dataset_structure(
        source_path=raw_data_path,
        target_binary_class="Painting",
        output_path=f"/tf/projet/Dataset_binary_{target_binary_class_name}"
    )

# 1. Check dataset existence
print(f"--Checking directory: {raw_data_path}--")
if not os.path.isdir(raw_data_path):
    print(f"Directory {raw_data_path} doesn't exist")
    exit(1)
else:
    print(f"Directory {raw_data_path} exists")

# 2. Fix data formats (if necessary)
if data_format_fix:
    data_formats_fixes(raw_data_path=raw_data_path)
subfolders = [f for f in os.listdir(raw_data_path) if os.path.isdir(os.path.join(raw_data_path, f)) and f not in excluded_data_folders]


# 3. Assemble dataset
dataset, class_names = dataset_assembly(
    raw_data_path=raw_data_path,
    subfolders=subfolders,
    batch_size=batch_size,
    img_height=img_height,
    img_width=img_width,
    seed=seed
)


# 4. Split dataset
train_ds, val_ds, test_ds = dataset_split(
    dataset=dataset,
    train_split=train_split,
    val_split=val_split,
    batch_size=batch_size,
    seed=seed
)


# 5. Generate model name
model_name = generate_model_name(
    project_name=project_name,
    model_arch=model_arch,
    target_class=target_binary_class_name if target_binary_class_name else None,
    transfer_learning=transfer_learning,
    epoch=epochs,
    batch_size=batch_size
)

--Checking directory: /tf/projet/Dataset--
Directory /tf/projet/Dataset exists
--Starting dataset assembly--
Found 41376 files belonging to 5 classes.
Detected classes: ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
Images batch shape : (32, 180, 180, 3)
Dataset assembly completed.
--Starting dataset split--
Dataset size: 1293
Creating training set of size ~33088...
Creating validation set of size ~4128...
Creating test set of size ~4160...
Dataset split completed.


2025-04-11 12:12:23.012913: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [35]:
## DATA VISUALIZATION FUNCTIONS
# Sample visualization
def visualize_class_samples(dataset, class_names, samples_per_class=5):
    """
    Visualize random samples from each class in the dataset.
    Parameters:
    - dataset: TensorFlow dataset
    - class_names: List of class names
    - samples_per_class: Number of samples to display per class
    """
    plt.figure(figsize=(15, 10))

    class_samples = {class_name: [] for class_name in class_names}

    for images, labels in dataset:
        for i, label in enumerate(labels.numpy()):
            class_name = class_names[label]
            if len(class_samples[class_name]) < samples_per_class:
                class_samples[class_name].append(images[i].numpy().astype("uint8"))

        if all(len(samples) >= samples_per_class for samples in class_samples.values()):
            break

    for idx, class_name in enumerate(class_names):
        for i, sample in enumerate(class_samples[class_name]):
            plt.subplot(len(class_names), samples_per_class, idx * samples_per_class + i + 1)
            plt.imshow(sample)
            plt.axis('off')
            if i == 0:
                plt.title(class_name)

    plt.tight_layout()
    plt.show()

# Visualize class distribution
def visualize_class_distribution(dataset, class_names):
    """
    Visualize the distribution of classes in the dataset.
    Parameters:
    - dataset: TensorFlow dataset
    - class_names: List of class names
    """
    class_counts = {class_name: 0 for class_name in class_names}

    for _, labels in dataset:
        for label in labels.numpy():
            class_counts[class_names[label]] += 1

    plt.figure(figsize=(12, 6))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title('Class Distribution')
    plt.xticks(rotation=45)
    plt.show()

In [36]:
## DATA VISUALIZATION WORKFLOW
if data_visualization:
    # 1. Visualize class samples
    visualize_class_samples(dataset=train_ds, class_names=class_names, samples_per_class=5)

    # 2. Visualize class distribution
    visualize_class_distribution(dataset=train_ds, class_names=class_names)

In [37]:
## CALLBACKS
class ConfusionMatrixCallback(tf.keras.callbacks.Callback):
    """
    TODO DOCSTRING
    """
    def __init__(self, val_data, class_names):
        super().__init__()
        self.val_data = val_data
        self.class_names = class_names

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []

        # Collect true labels and predictions
        for images, labels in self.val_data:
            preds = self.model.predict(images)
            y_true.extend(labels.numpy())
            y_pred.extend(np.argmax(preds, axis=1))

        # Generate confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(6, 6))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=self.class_names)
        disp.plot(ax=ax, xticks_rotation=45)
        plt.title(f"Confusion Matrix - Epoch {epoch + 1}")
        plt.tight_layout()

        # Log the confusion matrix as an image in wandb
        wandb.log({f"Confusion Matrix (Epoch {epoch + 1})": wandb.Image(fig)}, step=epoch)
        plt.close(fig)

        # Log the raw confusion matrix data for interactive dashboards in wandb
        wandb.log({f"Confusion Matrix Data (Epoch {epoch + 1})": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.class_names
        )}, step=epoch)


def create_callbacks(model_name="default_model", tensorboard=True, early_stopping=True, model_checkpoint=True, conf_matrix=False, val_data=None, class_names=None):
    """
    TODO DOCTSTRING
    """
    log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    callbacks = []

    if tensorboard:
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
        callbacks.append(tensorboard_callback)

    if early_stopping:
        early_stopping_callback = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=4,
            restore_best_weights=True
        )
        callbacks.append(early_stopping_callback)

    if model_checkpoint:
        checkpoint_dir = "checkpoints"
        os.makedirs(checkpoint_dir, exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(checkpoint_dir, f"{model_name}{timestamp}.keras"),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
        callbacks.append(model_checkpoint_callback)

    if conf_matrix and val_data is not None and class_names is not None:
        cm_callback = ConfusionMatrixCallback(val_data=val_data, class_names=class_names)
        callbacks.append(cm_callback)

    return callbacks

In [39]:
## MODELS PREPARATION FUNCTIONS
# Create model
def create_model(model_name, input_shape=(img_height, img_width, 3), num_classes=len(class_names), trainable=False, transfer_learning=False):
    """
    Creates an image classification model with customizable architecture.

    Args:
        model_name (str): Name of the model to create
        input_shape (tuple): Dimensions of input images (height, width, channels)
        num_classes (int): Number of classes for classification
        trainable (bool): If True, pre-trained model layers will be trainable
        transfer_learning (bool): If True, uses VGG16 pre-trained model as base

    Returns:
        tf.keras.Model: Model ready for training

    Notes:
        - When using transfer learning, the function creates a VGG16 base with custom top layers
        - When transfer_learning=False, creates a simple custom CNN architecture
        - GlobalAveragePooling2D is used instead of Flatten for transfer learning to reduce parameters
    """
    print(f"--Creating model: {model_name}--")

    if transfer_learning:
        # Base model creation with VGG16
        base_model = VGG16(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
        base_model.trainable = trainable

        # Custom layer on top of the base model
        inputs = tf.keras.Input(shape=input_shape)
        x = base_model(inputs, training=None)
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

        model = tf.keras.Model(inputs, outputs, name=model_name)

    else:
        # Custom model creation
        model = tf.keras.Sequential([
            tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(num_classes, activation='softmax')
        ], name=model_name)

    return model



def check_and_compile_model(model, model_name):
    """
    Check if model is compiled and compile it with default settings if not.
    Parameters:
    - model: The model to check
    - model_name: Name of the model (for logging)
    Returns:
    - The (possibly compiled) model
    """
    if not hasattr(model, 'optimizer') or model.optimizer is None:
        print(f"Model {model_name} is not compiled. Compiling with default settings...")
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
    return model

def train_model(model, model_name, train_ds, val_ds, epochs=10, save_path=None, class_weight= False):
    """
    Train a single model and optionally save it.

    Args:
        model: The model object to train
        model_name: Name of the model (for logging and saving)
        train_ds: Training dataset
        val_ds: Validation dataset
        epochs: Number of epochs to train the model
        save_path: Path to save the trained model (if None, model won't be saved)

    Returns:
        The trained model
    """
    print(f"\n--Training {model_name}--")

    model = check_and_compile_model(model, model_name)
    if (class_weight):
        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=2,
            class_weight = add_class_weights()
        )

    else :
        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=2
        )

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        model_save_path = os.path.join(save_path, f"{model_name}.keras")
        model.save(model_save_path)
        print(f"Model saved to {model_save_path}")

    return model


def add_class_weights():
    y_train = []

    for _, labels in train_ds:
        y_train.extend(labels.numpy())

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )

    class_weights_dict = dict(enumerate(class_weights))
    print("Class weights :", class_weights_dict)
    return class_weights_dict


In [35]:
## MODELS WORKFLOW
# 1. Create model
model = create_model(
    model_name=model_name,
    input_shape=(img_height, img_width, 3),
    num_classes=len(class_names),
    trainable=trainable,
    transfer_learning=transfer_learning
))

# 2. Initialize callbacks
run = wandb.init(
    entity="tom-antoine-cesi",
    project="Leyanda",
    name=f"model_comparison_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}",
    reinit=True,
    config={
        "models": list(models.keys()),
        "learning_rate": learning_rate,
        "epochs": epochs,
    }
)
callbacks_dict = {}
for model_name, model in models.items():
    callbacks_dict[model_name] = [WandbMetricsLogger()] + create_callbacks(
        model_name=model_name,
        tensorboard=True,
        early_stopping=True,
        model_checkpoint=True,
        conf_matrix=True,
        val_data=test_ds,
        class_names=class_names
    )

# 2. Train models
# train_models(
#     models=models,
#     train_ds=train_ds,
#     val_ds=val_ds,
#     epochs=epochs,
#     save_path=None,
#     history_save_path=history_save_path
# )

--Loading models from /tf/projet/Livrable 1/models--
Loading model: CNN_Dropout_3_0.3_BatchNormalization.keras
Successfully loaded model: CNN_Dropout_3_0.3_BatchNormalization
Loading model: CNN_Dropout_4_0.3.keras
Successfully loaded model: CNN_Dropout_4_0.3
Loaded 2 models: ['CNN_Dropout_3_0.3_BatchNormalization', 'CNN_Dropout_4_0.3']


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



--Training CNN_Dropout_3_0.3_BatchNormalization--
Epoch 1/10


Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


1034/1034 - 65s - 63ms/step - accuracy: 0.6851 - loss: 0.7612 - val_accuracy: 0.8086 - val_loss: 0.4774
Epoch 2/10


Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
2025-04-11 09:09:54.732603: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 12441856 bytes after encountering the first element of size 12441856 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


1034/1034 - 66s - 64ms/step - accuracy: 0.8121 - loss: 0.4531 - val_accuracy: 0.8622 - val_loss: 0.3649
Epoch 3/10


Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


1034/1034 - 66s - 64ms/step - accuracy: 0.8482 - loss: 0.3734 - val_accuracy: 0.8636 - val_loss: 0.3440
Epoch 4/10


KeyboardInterrupt: 